In [ ]:
#| default_exp preview

# Preview

> File preview extraction — returns the first N non-empty lines from text, PDF, DOCX, XLSX files, or an image marker.

In [ ]:
#| export
from pathlib import Path
from dataclasses import dataclass
import subprocess, shutil

In [ ]:
#| export
@dataclass
class PreviewResult:
    """Result of a file preview extraction."""
    lines: list[str]
    is_image: bool = False
    error: str | None = None

In [ ]:
#| export
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.gif', '.webp', '.bmp', '.svg', '.ico', '.tiff', '.tif'}
TEXT_EXTS = {'.txt', '.md', '.markdown', '.rst', '.csv', '.tsv', '.log', '.json', '.yaml', '.yml',
             '.toml', '.ini', '.cfg', '.conf', '.xml', '.html', '.htm', '.css', '.js', '.ts',
             '.py', '.sh', '.bash', '.zsh', '.ps1', '.bat', '.cmd', '.sql', '.r', '.jl'}

In [ ]:
#| export
def _take_lines(text: str, n: int) -> list[str]:
    """Extract first `n` non-empty lines from `text`."""
    lines = []
    for line in text.splitlines():
        stripped = line.strip()
        if stripped:
            lines.append(stripped)
            if len(lines) >= n: break
    return lines

In [ ]:
#| export
def _preview_text(path: Path, n: int) -> PreviewResult:
    """Preview a text-based file."""
    try:
        text = path.read_text(encoding='utf-8', errors='replace')
        return PreviewResult(lines=_take_lines(text, n))
    except Exception as e:
        return PreviewResult(lines=[], error=str(e))

def _preview_pdf(path: Path, n: int) -> PreviewResult:
    """Preview a PDF using pdfplumber."""
    try:
        import pdfplumber
        lines = []
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text = page.extract_text() or ""
                for line in text.splitlines():
                    stripped = line.strip()
                    if stripped:
                        lines.append(stripped)
                        if len(lines) >= n: return PreviewResult(lines=lines)
        return PreviewResult(lines=lines)
    except Exception as e:
        return PreviewResult(lines=[], error=str(e))

def _preview_docx(path: Path, n: int) -> PreviewResult:
    """Preview a DOCX using python-docx."""
    try:
        from docx import Document
        doc = Document(path)
        lines = []
        for para in doc.paragraphs:
            text = para.text.strip()
            if text:
                lines.append(text)
                if len(lines) >= n: break
        return PreviewResult(lines=lines)
    except Exception as e:
        return PreviewResult(lines=[], error=str(e))

def _preview_epub(path: Path, n: int) -> PreviewResult:
    """Preview an EPUB by converting to markdown via pandoc — first `n` non-empty lines."""
    if not shutil.which('pandoc'):
        return PreviewResult(lines=[], error="pandoc not installed (apt install pandoc)")
    try:
        res = subprocess.run(['pandoc', '-f', 'epub', '-t', 'markdown', str(path)],
                             capture_output=True, text=True, timeout=30)
        if res.returncode != 0: return PreviewResult(lines=[], error=res.stderr.strip() or 'pandoc failed')
        return PreviewResult(lines=_take_lines(res.stdout, n))
    except Exception as e:
        return PreviewResult(lines=[], error=str(e))

def _preview_xlsx(path: Path, n: int) -> PreviewResult:
    """Preview an XLSX — first sheet, first N non-empty rows as tab-separated."""
    try:
        from openpyxl import load_workbook
        wb = load_workbook(path, read_only=True, data_only=True)
        ws = wb.active
        lines = []
        for row in ws.iter_rows(values_only=True):
            vals = [str(c) if c is not None else "" for c in row]
            line = "\t".join(vals).strip()
            if line:
                lines.append(line)
                if len(lines) >= n: break
        wb.close()
        return PreviewResult(lines=lines)
    except Exception as e:
        return PreviewResult(lines=[], error=str(e))

In [ ]:
#| export
def get_preview(path: Path, n: int = 20) -> PreviewResult:
    """Get a preview of `path` — dispatches based on file extension."""
    suffix = path.suffix.lower()
    if suffix in IMAGE_EXTS:
        return PreviewResult(lines=[f"\ud83d\udcf7 Image: {path.name}", f"Open with [o] key"], is_image=True)
    if suffix == '.pdf': return _preview_pdf(path, n)
    if suffix == '.docx': return _preview_docx(path, n)
    if suffix == '.epub': return _preview_epub(path, n)
    if suffix in ('.xlsx', '.xls'): return _preview_xlsx(path, n)
    # Default: try as text
    return _preview_text(path, n)

## Tests

In [ ]:
from fastcore.test import *
import tempfile, os

# Test text preview
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write("\n\nLine 1\n\nLine 2\nLine 3\n\n")
    f.flush()
    result = get_preview(Path(f.name), n=2)
    test_eq(result.lines, ["Line 1", "Line 2"])
    test_eq(result.is_image, False)
    test_eq(result.error, None)
    os.unlink(f.name)

# Test empty file
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write("")
    f.flush()
    result = get_preview(Path(f.name))
    test_eq(result.lines, [])
    test_eq(result.error, None)
    os.unlink(f.name)

# Test image detection
result = get_preview(Path("photo.jpg"))
test_eq(result.is_image, True)
assert "Image" in result.lines[0]

result = get_preview(Path("screenshot.PNG"))
test_eq(result.is_image, True)

# Test markdown file (treated as text)
with tempfile.NamedTemporaryFile(mode='w', suffix='.md', delete=False) as f:
    f.write("# Title\n\nSome content\n\n## Section\n")
    f.flush()
    result = get_preview(Path(f.name), n=3)
    test_eq(result.lines, ["# Title", "Some content", "## Section"])
    os.unlink(f.name)

# Test _take_lines helper
test_eq(_take_lines("a\n\nb\nc", 2), ["a", "b"])
test_eq(_take_lines("", 5), [])
test_eq(_take_lines("\n\n\n", 5), [])

print("\u2713 All preview tests passed")